In [ ]:
!pip -q install transformers datasets accelerate scikit-learn pandas numpy evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


In [ ]:
import inspect
import json
import os
import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

MODEL_NAME = 'xlm-roberta-base'
MAX_LENGTH = 128
THRESHOLD = 0.5
INAPPROPRIATE_THRESHOLD = 0.65
CLEAN_MARGIN = 0.08
VAL_SIZE = 0.1
TEST_SIZE = 0.1
SEED = 42

BASE_DIR = Path('/content/drive/MyDrive/moderation_project')

OUTPUT_DIR = BASE_DIR / 'outputs'
MODEL_DIR = OUTPUT_DIR / 'model'
DATASET_PATH = BASE_DIR / 'moderation_dataset.csv'
TRAINER_STATE_PATH = OUTPUT_DIR / 'trainer_state.json'
TRAINING_LOG_PATH = OUTPUT_DIR / 'training_log.csv'
METRICS_PATH = OUTPUT_DIR / 'metrics.json'
LABELS_PATH = OUTPUT_DIR / 'label_names.json'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("DATASET_PATH:", DATASET_PATH)


Mounted at /content/drive
BASE_DIR: /content/drive/MyDrive/moderation_project
OUTPUT_DIR: /content/drive/MyDrive/moderation_project/outputs
MODEL_DIR: /content/drive/MyDrive/moderation_project/outputs/model
DATASET_PATH: /content/drive/MyDrive/moderation_project/moderation_dataset.csv


In [ ]:
import shutil

# Create a zip archive of the model directory
zip_file_path = OUTPUT_DIR / 'moderation_model.zip'
shutil.make_archive(str(zip_file_path.with_suffix('')), 'zip', MODEL_DIR)
print(f"Model archived to: {zip_file_path}")

Model archived to: /content/drive/MyDrive/moderation_project/outputs/moderation_model.zip


In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_file_path))

The model archive (`moderation_model.zip`) should now be downloaded to your local machine. If not, you can locate it in the Colab file browser at `/content/drive/MyDrive/moderation_project/outputs/moderation_model.zip` and download it manually.

In [ ]:
def normalize_text(text: str) -> str:
    text = str(text).strip().lower()
    text = re.sub(r'\s+', ' ', text)
    return text

def parse_labels(value: str):
    return [part.strip() for part in str(value).split('|') if part.strip()]

df = pd.read_csv(DATASET_PATH)
required_columns = {'text', 'labels'}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f'Missing columns: {missing}')

df = df[['text', 'labels']].dropna().copy()
df['text'] = df['text'].map(normalize_text)
df['labels'] = df['labels'].map(parse_labels)
df = df[df['text'].str.len() > 0]
df = df[df['labels'].map(len) > 0].reset_index(drop=True)

print('Rows:', len(df))
df.head()

Rows: 200000


,text,labels
0,kế hoạch này hơi khó hiểu.,[clean]
1,"new message: ""bấm vào link này để nhận quà""",[spam]
2,kiếm tiền online không cần vốn inbox ngay !!!,[spam]
3,vui lòng kiểm tra xem dự án này cần thêm chi t...,[clean]
4,"tin nhắn vừa nhận: ""i think this project is ac...",[clean]


In [ ]:
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['labels'])
label_names = list(mlb.classes_)
print('Labels:', label_names)

train_val_texts, test_texts, train_val_labels, test_labels = train_test_split(
    df['text'].tolist(),
    y,
    test_size=TEST_SIZE,
    random_state=SEED,
)

val_ratio_in_train_val = VAL_SIZE / (1 - TEST_SIZE)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_val_texts,
    train_val_labels,
    test_size=val_ratio_in_train_val,
    random_state=SEED,
)

print('Train size:', len(train_texts))
print('Val size:', len(val_texts))
print('Test size:', len(test_texts))

Labels: ['clean', 'hate', 'sexual', 'spam', 'toxic']
Train size: 160000
Val size: 20000
Test size: 20000


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ModerationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt',
        )
        item = {key: value.squeeze(0) for key, value in encoded.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

train_dataset = ModerationDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
val_dataset = ModerationDataset(val_texts, val_labels, tokenizer, MAX_LENGTH)
test_dataset = ModerationDataset(test_texts, test_labels, tokenizer, MAX_LENGTH)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= THRESHOLD).astype(int)

    return {
        'micro_f1': f1_score(labels, preds, average='micro', zero_division=0),
        'macro_f1': f1_score(labels, preds, average='macro', zero_division=0),
        'micro_precision': precision_score(labels, preds, average='micro', zero_division=0),
        'micro_recall': recall_score(labels, preds, average='micro', zero_division=0),
    }

In [ ]:
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    problem_type='multi_label_classification',
    id2label=id2label,
    label2id=label2id,
)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
training_kwargs = {
    'output_dir': str(OUTPUT_DIR / 'checkpoints'),
    'learning_rate': 2e-5,
    'per_device_train_batch_size': 16,
    'per_device_eval_batch_size': 16,
    'num_train_epochs': 3,
    'weight_decay': 0.01,
    'save_strategy': 'epoch',
    'save_total_limit': 2,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'micro_f1',
    'greater_is_better': True,
    'fp16': torch.cuda.is_available(),
    'logging_dir': str(OUTPUT_DIR / 'logs'),
    'logging_strategy': 'steps',
    'logging_steps': 20,
    'report_to': 'none',
    'seed': SEED,
}

signature = inspect.signature(TrainingArguments.__init__)
if 'evaluation_strategy' in signature.parameters:
    training_kwargs['evaluation_strategy'] = 'epoch'
else:
    training_kwargs['eval_strategy'] = 'epoch'

training_args = TrainingArguments(**training_kwargs)

trainer_kwargs = {
    'model': model,
    'args': training_args,
    'train_dataset': train_dataset,
    'eval_dataset': val_dataset,
    'compute_metrics': compute_metrics,
}

trainer_signature = inspect.signature(Trainer.__init__)
if 'processing_class' in trainer_signature.parameters:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in trainer_signature.parameters:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = Trainer(**trainer_kwargs)

train_result = trainer.train()
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
trainer.save_state()
trainer.state.save_to_json(str(TRAINER_STATE_PATH))

val_metrics = trainer.evaluate(eval_dataset=val_dataset, metric_key_prefix='val')
test_metrics = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix='test')

with open(LABELS_PATH, 'w', encoding='utf-8') as fp:
    json.dump(label_names, fp, ensure_ascii=False, indent=2)

metrics_payload = {
    'train': train_result.metrics,
    'val': val_metrics,
    'test': test_metrics,
}

with open(METRICS_PATH, 'w', encoding='utf-8') as fp:
    json.dump(metrics_payload, fp, ensure_ascii=False, indent=2)

pd.DataFrame(trainer.state.log_history).to_csv(TRAINING_LOG_PATH, index=False)

print('Saved model to:', MODEL_DIR)
print('Saved trainer state to:', TRAINER_STATE_PATH)
print('Saved training log to:', TRAINING_LOG_PATH)
print('Saved metrics to:', METRICS_PATH)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Micro Precision,Micro Recall
1,0.000080,0.000068,1.000000,1.000000,1.000000,1.000000
2,0.000015,0.000013,1.000000,1.000000,1.000000,1.000000
3,0.000008,0.000007,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to: /content/drive/MyDrive/moderation_project/outputs/model
Saved trainer state to: /content/drive/MyDrive/moderation_project/outputs/trainer_state.json
Saved training log to: /content/drive/MyDrive/moderation_project/outputs/training_log.csv
Saved metrics to: /content/drive/MyDrive/moderation_project/outputs/metrics.json


In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load model đã train
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

with open(LABELS_PATH, "r", encoding="utf-8") as f:
    label_names = json.load(f)

def predict_text(text, threshold=0.5):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
    )

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.sigmoid(logits)[0].cpu().numpy()

    results = [
        (label_names[i], float(probs[i]))
        for i in range(len(label_names))
    ]
    predicted = [label for label, prob in results if prob >= threshold]
    return predicted, results

  # Test thử
text = "bấm vào link này để nhận quà miễn phí ngay"
predicted, scores = predict_text(text, threshold=THRESHOLD)

print("Text:", text)
print("Predicted labels:", predicted)
print("Scores:")
for label, score in scores:
    print(f"  {label}: {score:.4f}")

samples = [
    "bấm vào link này để nhận quà miễn phí ngay",
    "dự án này cần bổ sung thêm tài liệu kỹ thuật",
    "mày ngu thế",
]

for text in samples:
    predicted, _ = predict_text(text, threshold=THRESHOLD)
    print(f"{text} -> {predicted}")


for i in range(10):
    predicted, _ = predict_text(test_texts[i], threshold=THRESHOLD)
    true_labels = [label_names[j] for j, v in enumerate(test_labels[i]) if v == 1]
    print("TEXT :", test_texts[i])
    print("TRUE :", true_labels)
    print("PRED :", predicted)
    print("-" * 50)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Text: bấm vào link này để nhận quà miễn phí ngay
Predicted labels: ['spam']
Scores:
  clean: 0.0001
  hate: 0.0001
  sexual: 0.0001
  spam: 0.9999
  toxic: 0.0001
bấm vào link này để nhận quà miễn phí ngay -> ['spam']
dự án này cần bổ sung thêm tài liệu kỹ thuật -> ['clean']
mày ngu thế -> ['toxic']


NameError: name 'test_texts' is not defined